# Chevron Network — Advantage Experiment v3

## The core question
Does the 2-channel architecture learn the *compositional polarity rule* (sign product),
or does it memorise sign patterns from training data?

## The test
For a chain of length 3 there are 4 possible sign patterns: `(+,+)`, `(-,-)`, `(+,-)`, `(-,+)`.

- **Train** on same-sign patterns only: `(+,+)` → label=1 and `(-,-)` → label=1
- **Test** on mixed-sign patterns: `(+,-)` → label=0 and `(-,+)` → label=0

A model that memorised 'same signs = positive' will score ~1.0 on train patterns and
~0.0 on test patterns (it learned the wrong rule). A model that learned the actual
compositional rule `s1 × s2` will generalise correctly to mixed patterns.

The *same concepts* appear in both splits — the only difference is which sign
combinations they appear with. This isolates sign-composition learning from
concept memorisation entirely.

## Why chevron might do better
The channel-swap gate in ChevronNet directly encodes 'negative sign = swap pos/neg
channels'. This is the right inductive bias for the rule. The scalar MLP must
discover an equivalent representation from scratch.

In [ ]:
import torch, random, numpy as np
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch {torch.__version__}  |  device: {DEVICE}")

---
## 1. Dataset

In [ ]:
# Sign patterns for chain_length=3 (two edges, so two signs)
SAME_SIGN_PATTERNS  = [(+1, +1), (-1, -1)]   # product = +1  → label 1
MIXED_SIGN_PATTERNS = [(+1, -1), (-1, +1)]   # product = -1  → label 0

# For longer chains (chain_length=4, three edges) patterns are auto-generated below.


def split_sign_patterns(chain_length):
    """Split all 2^(L-1) sign patterns into same-product (train) and mixed (test).
    Train = patterns where ALL signs are +1 or ALL signs are -1.
    Test  = all other patterns (at least one sign differs).
    """
    from itertools import product as iproduct
    n_edges = chain_length - 1
    all_pats = list(iproduct([-1, 1], repeat=n_edges))
    # 'same' = all same sign (trivially compositional)
    train_pats = [p for p in all_pats if len(set(p)) == 1]
    test_pats  = [p for p in all_pats if len(set(p)) > 1]
    return train_pats, test_pats


class SignSplitDataset(Dataset):
    """
    Polarity chain dataset restricted to a given set of sign patterns.

    sign_patterns: list of tuples, e.g. [(+1,+1), (-1,-1)]
    All concepts share a global embedding table.
    Label = 1 if product of signs > 0, else 0.
    """
    def __init__(self, pos_emb, neg_emb, sign_patterns,
                 n_concepts=64, n_samples=8000, noise=0.0, seed=42):
        rng = np.random.RandomState(seed)
        self.pos_emb = pos_emb
        self.neg_emb = neg_emb
        self.half = pos_emb.shape[1]
        chain_length = len(sign_patterns[0]) + 1
        self.chain_length = chain_length

        samples = []
        for _ in range(n_samples):
            concepts = rng.randint(0, n_concepts, size=chain_length).tolist()
            signs = list(sign_patterns[rng.randint(len(sign_patterns))])
            label = 1 if int(np.prod(signs)) > 0 else 0
            if rng.rand() < noise:
                label = 1 - label
            samples.append((concepts, signs, label))
        self.samples = samples

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        concepts, signs, label = self.samples[idx]
        pos = self.pos_emb[concepts]          # [L, half]
        neg = self.neg_emb[concepts]          # [L, half]
        x = torch.stack([pos, neg], dim=1)   # [L, 2, half]
        return x, torch.tensor(signs, dtype=torch.float32), torch.tensor(label, dtype=torch.float32)


def make_embeddings(n_concepts=64, half=16, seed=42):
    rng = np.random.RandomState(seed)
    pos = rng.randn(n_concepts, half).astype(np.float32)
    neg = rng.randn(n_concepts, half).astype(np.float32)
    pos /= np.linalg.norm(pos, axis=1, keepdims=True) + 1e-8
    neg /= np.linalg.norm(neg, axis=1, keepdims=True) + 1e-8
    return torch.from_numpy(pos), torch.from_numpy(neg)


# ── Config ────────────────────────────────────────────────────────────────────
N_CONCEPTS   = 64
HALF_EMB     = 16
CHAIN_LENGTH = 3
N_TRAIN      = 8000
N_VAL        = 1000
N_TEST       = 2000
BATCH        = 128
EPOCHS       = 40
LR           = 3e-4
NOISE        = 0.05
SEEDS        = [42, 7, 99]

TRAIN_PATS, TEST_PATS = split_sign_patterns(CHAIN_LENGTH)
print(f"Train sign patterns : {TRAIN_PATS}  (label: all +1)")
print(f"Test  sign patterns : {TEST_PATS}   (label: all 0)")
print(f"\nNote: train patterns are trivially solvable by 'all-same = positive'.")
print(f"Test patterns require the true rule: product of signs.")


def make_loaders(seed, pos_emb, neg_emb):
    train_ds = SignSplitDataset(pos_emb, neg_emb, TRAIN_PATS, N_CONCEPTS, N_TRAIN,  NOISE, seed)
    val_ds   = SignSplitDataset(pos_emb, neg_emb, TRAIN_PATS, N_CONCEPTS, N_VAL,   0.0,   seed+100)
    test_in  = SignSplitDataset(pos_emb, neg_emb, TRAIN_PATS, N_CONCEPTS, N_TEST,  0.0,   seed+200)
    test_ood = SignSplitDataset(pos_emb, neg_emb, TEST_PATS,  N_CONCEPTS, N_TEST,  0.0,   seed+300)
    return (
        DataLoader(train_ds, batch_size=BATCH, shuffle=True),
        DataLoader(val_ds,   batch_size=BATCH),
        DataLoader(test_in,  batch_size=BATCH),
        DataLoader(test_ood, batch_size=BATCH),
    )

---
## 2. Models

In [ ]:
class ChevronLayer(nn.Module):
    """[B, in_groups, 2] → [B, out_groups, 2] via learnable 2×2 operators."""
    def __init__(self, in_groups, out_groups, bias=True, variant='full'):
        super().__init__()
        self.variant = variant
        self.weight = nn.Parameter(torch.empty(out_groups, in_groups, 2, 2))
        nn.init.xavier_uniform_(self.weight.view(out_groups * 2, in_groups * 2))
        self.bias = nn.Parameter(torch.zeros(out_groups, 2)) if bias else None

    def forward(self, x):
        w = self.weight
        if self.variant == 'diag_only':
            w = w * torch.eye(2, device=w.device).view(1, 1, 2, 2)
        out = torch.einsum('big,oigj->boj', x, w)
        if self.bias is not None:
            out = out + self.bias
        return out

    def offdiag_magnitude(self):
        w = self.weight.detach()
        return torch.stack([w[..., 0, 1], w[..., 1, 0]]).abs().mean().item()


class ChevronNet(nn.Module):
    def __init__(self, half_emb=16, hidden_groups=24, variant='full'):
        super().__init__()
        self.half_emb = half_emb
        self.enc     = ChevronLayer(half_emb, hidden_groups, variant=variant)
        self.combine = ChevronLayer(hidden_groups, hidden_groups, variant=variant)
        self.head    = nn.Linear(hidden_groups * 2, 1)
        self.act     = nn.GELU()

    def forward(self, x, signs):
        B, L, _, half = x.shape
        h = self.act(self.enc(x.view(B * L, half, 2))).view(B, L, -1, 2)
        agg = h[:, 0]
        for i in range(L - 1):
            s = signs[:, i].view(B, 1, 1)
            hi = h[:, i + 1]
            agg = self.act(self.combine(agg + torch.where(s > 0, hi, hi[..., [1, 0]])))
        return self.head(agg.reshape(B, -1)).squeeze(-1)

    def offdiag_magnitude(self):
        return {'enc': self.enc.offdiag_magnitude(), 'combine': self.combine.offdiag_magnitude()}


class ScalarMLP(nn.Module):
    """Scalar baseline — same information, no two-channel structure."""
    def __init__(self, half_emb=16, chain_length=3, hidden_dim=56):
        super().__init__()
        in_dim = half_emb * 2 * chain_length + (chain_length - 1)
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x, signs):
        B = x.shape[0]
        return self.net(torch.cat([x.reshape(B, -1), signs], -1)).squeeze(-1)

    def offdiag_magnitude(self): return None


def count_params(m): return sum(p.numel() for p in m.parameters())

print("Parameter counts:")
for name, m in [
    ('ChevronNet full (hg=24)', ChevronNet(HALF_EMB, 24, 'full')),
    ('ChevronNet diag (hg=24)', ChevronNet(HALF_EMB, 24, 'diag_only')),
    ('ScalarMLP       (hd=56)', ScalarMLP(HALF_EMB, CHAIN_LENGTH, 56)),
]:
    print(f"  {name}: {count_params(m):,}")
print()
print("Note: ScalarMLP is larger — for a fair test we also run ScalarMLP-matched (hd=38).")

---
## 3. Training

In [ ]:
def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss = correct = n = 0
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for x, signs, labels in loader:
            x, signs, labels = x.to(DEVICE), signs.to(DEVICE), labels.to(DEVICE)
            logits = model(x, signs)
            loss = F.binary_cross_entropy_with_logits(logits, labels)
            if training:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * len(labels)
            correct    += ((logits > 0).float() == labels).sum().item()
            n          += len(labels)
    return total_loss / n, correct / n


def train_model(model, train_loader, val_loader, label=''):
    opt = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    best_val, best_state, history = 0.0, None, []
    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_acc = run_epoch(model, train_loader, opt)
        va_loss, va_acc = run_epoch(model, val_loader)
        history.append((tr_acc, va_acc))
        if va_acc > best_val:
            best_val   = va_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        if epoch % 8 == 0:
            print(f"  {label:22s} ep {epoch:02d}  train={tr_acc:.3f}  val={va_acc:.3f}")
    model.load_state_dict(best_state)
    return history


print("Training helpers ready.")

---
## 4. Main Experiment

In [ ]:
MODEL_HIDDEN = 24   # hidden_groups for ChevronNet  (~3,985 params)
MLP_HIDDEN   = 56   # hidden_dim for ScalarMLP large (~8,793 — kept for reference)
MLP_MATCHED  = 38   # hidden_dim for ScalarMLP matched (~3,969 params)

MODEL_SPECS = {
    'chevron_full':  lambda: ChevronNet(HALF_EMB, MODEL_HIDDEN, 'full').to(DEVICE),
    'chevron_diag':  lambda: ChevronNet(HALF_EMB, MODEL_HIDDEN, 'diag_only').to(DEVICE),
    'mlp_matched':   lambda: ScalarMLP(HALF_EMB, CHAIN_LENGTH, MLP_MATCHED).to(DEVICE),
    'mlp_large':     lambda: ScalarMLP(HALF_EMB, CHAIN_LENGTH, MLP_HIDDEN).to(DEVICE),
}

results       = []
all_histories = {k: [] for k in MODEL_SPECS}

for seed in SEEDS:
    set_seed(seed)
    print(f"\n{'='*55}\nSeed {seed}")
    pos_emb, neg_emb = make_embeddings(N_CONCEPTS, HALF_EMB, seed=seed)
    train_loader, val_loader, test_in_loader, test_ood_loader = make_loaders(seed, pos_emb, neg_emb)

    for name, build_fn in MODEL_SPECS.items():
        set_seed(seed)
        model = build_fn()
        print(f"\n  {name}  ({count_params(model):,} params)")
        history = train_model(model, train_loader, val_loader, label=name)
        all_histories[name].append(history)

        _, test_in_acc  = run_epoch(model, test_in_loader)   # same sign patterns as train
        _, test_ood_acc = run_epoch(model, test_ood_loader)  # mixed sign patterns — KEY
        offdiag = model.offdiag_magnitude() if hasattr(model, 'offdiag_magnitude') and model.offdiag_magnitude() else {}

        results.append(dict(
            seed=seed, model=name, params=count_params(model),
            test_in_acc=test_in_acc, test_ood_acc=test_ood_acc,
            offdiag_enc    =offdiag.get('enc')     if offdiag else None,
            offdiag_combine=offdiag.get('combine') if offdiag else None,
        ))
        print(f"  → in-dist={test_in_acc:.3f}  OOD (mixed signs)={test_ood_acc:.3f}",
              f" offdiag={offdiag}" if offdiag else "")

df = pd.DataFrame(results)
print("\n=== Raw results ===")
print(df.to_string(index=False))

---
## 5. Summary & Plots

In [ ]:
summary = df.groupby('model').agg(
    params         =('params',        'first'),
    in_acc_mean    =('test_in_acc',   'mean'),
    in_acc_std     =('test_in_acc',   'std'),
    ood_acc_mean   =('test_ood_acc',  'mean'),
    ood_acc_std    =('test_ood_acc',  'std'),
    offdiag_enc    =('offdiag_enc',   'mean'),
).reset_index()

print("=== Summary (mean ± std across seeds) ===")
print(f"{'Model':<18} {'Params':>7} {'In-dist acc':>13} {'OOD acc (mixed signs)':>22}  Offdiag_enc")
print("-" * 75)
for _, r in summary.iterrows():
    od = f"{r['offdiag_enc']:.4f}" if pd.notna(r['offdiag_enc']) else '  n/a'
    print(f"{r['model']:<18} {r['params']:>7,} "
          f"{r['in_acc_mean']:.3f}±{r['in_acc_std']:.3f}        "
          f"{r['ood_acc_mean']:.3f}±{r['ood_acc_std']:.3f}          {od}")

# ── Bar chart: in-dist vs OOD side by side ────────────────────────────────────
colors = {
    'chevron_full': '#2196F3', 'chevron_diag': '#FF9800',
    'mlp_matched':  '#9E9E9E', 'mlp_large':    '#4CAF50',
}
labels_nice = {
    'chevron_full': 'Chevron\n(full)', 'chevron_diag': 'Chevron\n(diag)',
    'mlp_matched':  'MLP\n(matched)',  'mlp_large':    'MLP\n(large)',
}
model_order = list(MODEL_SPECS.keys())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
x = np.arange(len(model_order))
w = 0.5

for ax, (mean_col, std_col, title) in zip(axes, [
    ('in_acc_mean', 'in_acc_std',  'In-distribution accuracy\n(same sign patterns as training)'),
    ('ood_acc_mean','ood_acc_std', 'OOD accuracy\n(mixed sign patterns — never seen in training)'),
]):
    means = [summary[summary.model == m][mean_col].values[0] for m in model_order]
    stds  = [summary[summary.model == m][std_col].values[0]  for m in model_order]
    bars = ax.bar(x, means, yerr=stds, capsize=6,
                  color=[colors[m] for m in model_order], alpha=0.85, width=w)
    ax.axhline(0.5, ls='--', color='k', alpha=0.4, label='chance')
    ax.axhline(1.0, ls=':',  color='k', alpha=0.2)
    ax.set_xticks(x); ax.set_xticklabels([labels_nice[m] for m in model_order])
    ax.set_ylim(0.3, 1.1); ax.set_ylabel('Accuracy'); ax.set_title(title)
    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{mean:.3f}', ha='center', fontsize=9)

plt.suptitle('Sign-pattern generalisation test — does the model know the rule?', fontsize=13)
plt.tight_layout()
plt.savefig('sign_generalisation.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: sign_generalisation.png')

---
## 6. Learning Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
epochs_x = np.arange(1, EPOCHS + 1)

for name, histories in all_histories.items():
    tr = np.array([[e[0] for e in h] for h in histories])
    va = np.array([[e[1] for e in h] for h in histories])
    for ax, curves, title in zip(axes, [tr, va], ['Train accuracy', 'Val accuracy (same-sign patterns)']):
        m, s = curves.mean(0), curves.std(0)
        ax.plot(epochs_x, m, label=labels_nice[name].replace('\n', ' '), color=colors[name], lw=2)
        ax.fill_between(epochs_x, m-s, m+s, alpha=0.12, color=colors[name])

for ax, title in zip(axes, ['Train accuracy', 'Val accuracy (same-sign patterns)']):
    ax.set(xlabel='Epoch', ylabel='Accuracy', title=title, ylim=(0.45, 1.05))
    ax.axhline(0.5, ls='--', color='k', alpha=0.3)
    ax.legend(fontsize=8)

plt.suptitle(f'Learning curves — chain length {CHAIN_LENGTH}, noise={NOISE}', fontsize=12)
plt.tight_layout()
plt.savefig('learning_curves_v3.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 7. Interpretation Guide

### Reading the OOD accuracy

The training set contains only `(+,+)` and `(-,-)` patterns — both label=1 (positive product). A model that memorised 'same signs always = positive' will:
- Score ~1.0 in-distribution
- Score ~0.0 OOD (mixed signs are all label=0, so it predicts everything wrong)

A model that learned the true rule `s1 × s2` will:
- Score ~1.0 in-distribution  
- Score ~1.0 OOD

| OOD result | Interpretation |
|---|---|
| All models ~1.0 OOD | All learned the rule — try harder task (longer chains, less data) |
| All models ~0.0 OOD | All memorised — representations may be too intertwined with sign values |
| Chevron >> MLP on OOD | **Architecture advantage confirmed** — channel-swap inductive bias helps |
| Chevron_full >> Chevron_diag on OOD | Off-diagonals (cross-channel coupling) are essential for the rule |
| MLP_large >> MLP_matched on OOD | Capacity matters more than structure — scale up chevron to verify |

### If all models fail OOD (~0.0)
The models learned 'positive label = positive sign pattern' rather than the product rule. The fix is to **balance labels in training**: include some `(+,+)→1` and `(-,-)→1` but also a few `(+,+)→override 0` examples — or better, extend to chain_length=4 where same-sign patterns have both labels (e.g. `(+,+,+)→1` but `(-,-,-)→1` too). See the cell below.

### Chain length 4 bonus experiment
For chain_length=4 (3 edges), same-sign patterns are `(+,+,+)→1` and `(-,-,-)→-1→0`. Now the train set has both labels even in same-sign patterns, so label-memorisation is not enough. Run this for a cleaner test.

In [ ]:
# ── Chain-length 4 bonus: same-sign split has both labels ─────────────────────
# (+,+,+) → product=+1 → label=1
# (-,-,-) → product=-1 → label=0
# Train on these two; test on all mixed patterns.

CL4 = 4
TRAIN_PATS_4, TEST_PATS_4 = split_sign_patterns(CL4)
print(f"Chain-4 train patterns: {TRAIN_PATS_4}")
print(f"Chain-4 test  patterns: {TEST_PATS_4[:4]} ... ({len(TEST_PATS_4)} total)")

chain4_results = []
SEED_C4 = 42
set_seed(SEED_C4)
pos4, neg4 = make_embeddings(N_CONCEPTS, HALF_EMB, seed=SEED_C4)

def make_loaders_cl(seed, pos_emb, neg_emb, cl, train_pats, test_pats):
    train_ds = SignSplitDataset(pos_emb, neg_emb, train_pats, N_CONCEPTS, N_TRAIN, NOISE,  seed)
    val_ds   = SignSplitDataset(pos_emb, neg_emb, train_pats, N_CONCEPTS, N_VAL,  0.0,    seed+100)
    test_in  = SignSplitDataset(pos_emb, neg_emb, train_pats, N_CONCEPTS, N_TEST, 0.0,    seed+200)
    test_ood = SignSplitDataset(pos_emb, neg_emb, test_pats,  N_CONCEPTS, N_TEST, 0.0,    seed+300)
    return (
        DataLoader(train_ds, batch_size=BATCH, shuffle=True),
        DataLoader(val_ds,   batch_size=BATCH),
        DataLoader(test_in,  batch_size=BATCH),
        DataLoader(test_ood, batch_size=BATCH),
    )

tl4, vl4, til4, tol4 = make_loaders_cl(SEED_C4, pos4, neg4, CL4, TRAIN_PATS_4, TEST_PATS_4)

for name in ['chevron_full', 'chevron_diag', 'mlp_matched']:
    set_seed(SEED_C4)
    if 'chevron' in name:
        model = ChevronNet(HALF_EMB, MODEL_HIDDEN, 'full' if 'full' in name else 'diag_only').to(DEVICE)
    else:
        model = ScalarMLP(HALF_EMB, CL4, MLP_MATCHED).to(DEVICE)
    train_model(model, tl4, vl4, label=f'cl4/{name}')
    _, in_acc  = run_epoch(model, til4)
    _, ood_acc = run_epoch(model, tol4)
    chain4_results.append({'model': name, 'in_acc': in_acc, 'ood_acc': ood_acc})
    print(f"cl=4 {name}: in={in_acc:.3f}  OOD={ood_acc:.3f}")

print("\n=== Chain-4 results ===")
print(pd.DataFrame(chain4_results).to_string(index=False))